<a href="https://colab.research.google.com/github/Kanaklata91/Google-Colab/blob/master/notebooks/colab-github-demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Creating a sample gpt like feature , by training the transformers on a small set of data.



In [2]:
#downloaded the raw shakespeare dataset
# https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-05-26 08:33:08--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2026-05-26 08:33:08 (22.3 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [3]:
with open('input.txt','r', encoding='utf-8') as f :
  text = f.read()

In [4]:
print('length of dataset in characters ', len(text))

length of dataset in characters  1115394


In [5]:
#read through first 1000 characters
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [6]:
#print unique characters present in the data including spaces , commas , and alphabets and numbers

chars = sorted(list(set(text)))
vocab_size=len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [7]:
#next going to convert every character into number using encode and decode feature

stoi = { ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i, ch in enumerate(chars)}
encode = lambda s:[stoi[c] for c in s]
decode = lambda l:''.join(itos[i] for i in l)

print(encode('Hi There'))
print(decode(encode('Hi There')))

[20, 47, 1, 32, 46, 43, 56, 43]
Hi There


In [8]:
#using the pytorch library , encode entire data
import torch
data = torch.tensor(encode(text),dtype=torch.long)
print(data.shape , data.dtype)
print(data[:1000])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [9]:
#distribute the data in training and test dataset
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [10]:
# cannot feed entire text at once to transformer so have to split the data in chunks
block_size = 8
train_data[:block_size+1]


tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [11]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
  context = x[:t+1]
  target = y[t]
  print(f"when input is {context} the target is {target}")


when input is tensor([18]) the target is 47
when input is tensor([18, 47]) the target is 56
when input is tensor([18, 47, 56]) the target is 57
when input is tensor([18, 47, 56, 57]) the target is 58
when input is tensor([18, 47, 56, 57, 58]) the target is 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target is 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target is 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target is 58


In [12]:
torch.manual_seed(1337)
batch_size=4 # number of independent sequences will be processed in parallel
block_size=8  # max context length for predictions

def get_batch(split):
  data = train_data if split=='train' else val_data
  ix = torch.randint(len(data)-block_size,(batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  return x , y

xb , yb = get_batch('train')
print("inputs:")
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])


In [13]:
print(xb)

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


In [14]:
#implementation of Deep Learning , Bigram language model using pytorch library
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
  def __init__(self , vocab_size):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size , vocab_size)


  def forward(self , idx,targets=None):
    logits= self.token_embedding_table(idx)
    if targets is None:
      loss=None
    else:
     # logits = self.token_embedding_table(idx) #B - batch T - time C - channel tensor
      B,T,C = logits.shape
      logits = logits.view(B*T,C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits , targets)
    return logits , loss

  def generate(self , idx , max_new_tokens):
    for _ in range(max_new_tokens):
      logits, loss= self(idx)
      logits = logits[:,-1,:]
      #apply softmax to generate probabilities
      probs = F.softmax(logits,dim=-1)
      idx_next = torch.multinomial(probs , num_samples=1)
      idx = torch.cat((idx,idx_next),dim=1)
    return idx


m = BigramLanguageModel(vocab_size)
logits , loss  = m(xb,yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx=torch.zeros((1,1),dtype=torch.long),max_new_tokens=500)[0].tolist()))

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJt-wBpm&yiltNCjeO3:Cx&vvMYW-txjuAd IRFbTpJ$zkZelxZtTlHNzdXXUiQQY:qFINTOBNLI,&oTigq z.c:Cq,SDXzetn3XVjX-YBcHAUhk&PHdhcOb
nhJ?FJU?pRiOLQeUN!BxjPLiq-GJdUV'hsnla!murI!IM?SPNPq?VgC'R
pD3cLv-bxn-tL!upg
SZ!Uvdg CtxtT?hsiW:XxKIiPlagHIsr'zKSVxza?GlDWObPmRJgrIAcmspmZ&viCKot:u3qYXA:rZgv f:3Q-oiwUzqh'Z!I'zRS3SP rVchSFUIdd q?sPJpUdhMCK$VXXevXJFMl,i
YxA:gWId,EXR,iMC,$?srV$VztRwb?KpgUWFjR$zChOLm;JrDnDph
LBj,KZxJa


In [15]:
#create pytorch optimizer
optimizer = torch.optim.AdamW(m.parameters(),lr=1e-3)

In [16]:
batch_size =32
for steps in range(10000):
  xb , yb = get_batch('train')
  #evaluate the loss
  logits , loss = m(xb , yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()

print(loss.item())

2.4771134853363037


In [17]:
print(decode(m.generate(idx=torch.zeros((1,1),dtype=torch.long),max_new_tokens=500)[0].tolist()))


Thit wentem lud engitonso; cer ize helour
Jginte the?
Thak orblyoruldvicee chot, p,
Bealivolde Th likl beamen, tofr,
n s Byo tred ceathe, il ivilde w
O ff y
Five:
Mied aiMy, I ivis muofounce herevern outh f athawendesees yof th withS:

FiFLINR:

Wheader y blitow,
Ye m o ditoshyd me, ch rte u hart ararwsa
Wou fe,
INurathoune
IESSARin,
MIO:
Ind sust tl.
S:
NMy BAnind g.
iudshank
An chin is a arokisupxaseru t w ity merwo al LOLo bebte loolld worinero ya l aknge ond thal ttry b's mo ge ck.

gh, chee


#Mathematical trick on Self Attention


In [18]:
#sample example
# note : so there are 8 tokens 1,2,3,4,5,6,7,8 . So token 5 cannot communicate with 6,7,8 but should communicate with 1,2,3,4 . because 6,7,8 token is for the prediction
# so in order for tokens to communicate with each other , we need to vectorize the past and the present tokens by taking average of all tokens 1,2,3,4,5
torch.manual_seed(1337)
B,T,C = 4,8,2 #batch , time , channel
x=torch.randn(B,T,C)
x.shape



torch.Size([4, 8, 2])

In [19]:
xbow = torch.zeros((B,T,C))
for b in range(B):
  for t in range(T):
    xprev = x[b,:t+1]
    xbow[b,t] = torch.mean(xprev,0)

In [20]:
print(x[0])

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])


In [21]:
print(xbow[0])

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])


In [37]:
#self attention
torch.manual_seed(1337)
B,T,C = 4,8,32
x = torch.randn(B,T,C)

head_size = 16
key = nn.Linear(C ,head_size , bias=False)
query=nn.Linear(C , head_size, bias=False)
value = nn.Linear(C , head_size , bias=False)
k = key(x) #(B,T,16)
q=query(x)  #(B,T,16)
weights = q @ k.transpose(-2,-1) # (B,T,16) @ (B , 16 ,T) - > (B,T,T)

tril = torch.tril(torch.ones(T,T))
#weights= torch.zeros((T,T))
weights=weights.masked_fill(tril==0,float('-inf'))
weights=F.softmax(weights,dim=-1)
v=value(x)
out = weights @ x
out.shape

torch.Size([4, 8, 32])

In [35]:
weights[0]

tensor([[0.0078, 0.0124, 0.0803, 0.3961, 0.0157, 0.3248, 0.1338, 0.0290],
        [0.0010, 0.0053, 0.0309, 0.8170, 0.0031, 0.0790, 0.0264, 0.0373],
        [0.0803, 0.0633, 0.2408, 0.1524, 0.0834, 0.0534, 0.2405, 0.0859],
        [0.4135, 0.0848, 0.1349, 0.0808, 0.1079, 0.0586, 0.0519, 0.0677],
        [0.0178, 0.0635, 0.0284, 0.0167, 0.4779, 0.1479, 0.0905, 0.1574],
        [0.0149, 0.2277, 0.0182, 0.0075, 0.5770, 0.0016, 0.0841, 0.0689],
        [0.1500, 0.3607, 0.0389, 0.0369, 0.0929, 0.1785, 0.0292, 0.1130],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

In [36]:
weights

tensor([[[0.0078, 0.0124, 0.0803, 0.3961, 0.0157, 0.3248, 0.1338, 0.0290],
         [0.0010, 0.0053, 0.0309, 0.8170, 0.0031, 0.0790, 0.0264, 0.0373],
         [0.0803, 0.0633, 0.2408, 0.1524, 0.0834, 0.0534, 0.2405, 0.0859],
         [0.4135, 0.0848, 0.1349, 0.0808, 0.1079, 0.0586, 0.0519, 0.0677],
         [0.0178, 0.0635, 0.0284, 0.0167, 0.4779, 0.1479, 0.0905, 0.1574],
         [0.0149, 0.2277, 0.0182, 0.0075, 0.5770, 0.0016, 0.0841, 0.0689],
         [0.1500, 0.3607, 0.0389, 0.0369, 0.0929, 0.1785, 0.0292, 0.1130],
         [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],

        [[0.0336, 0.0118, 0.2053, 0.0533, 0.3595, 0.2894, 0.0404, 0.0066],
         [0.0017, 0.0082, 0.0281, 0.3554, 0.0286, 0.0429, 0.4809, 0.0542],
         [0.0624, 0.0130, 0.1766, 0.0688, 0.2372, 0.4006, 0.0216, 0.0198],
         [0.3046, 0.0661, 0.2588, 0.0613, 0.0123, 0.0019, 0.0091, 0.2859],
         [0.0041, 0.0274, 0.0180, 0.4643, 0.0855, 0.0511, 0.3014, 0.0482],
         [0.0086, 0.011